# 08 — Hero-Specific Events: Mercy Rez, D.Va Remech, Echo Duplicate

Some Overwatch heroes have unique mechanics that generate their own event tables in the Parsertime dataset. These are among the most interesting and least-analyzed aspects of competitive OW:

### What We Analyze

1. **Mercy Resurrect** (`MercyRez` table): Mercy's Resurrect is one of the most impactful abilities in Overwatch — it literally reverses a kill. We analyze when rezzes happen, who gets rezzed, and how they affect fight outcomes.

2. **D.Va Remech** (`DvaRemech` + `RemechCharged` tables): When D.Va's mech is destroyed, she enters "pilot form" — a vulnerable state. Getting remech back (either through charging it or calling it) is a critical survival moment. We analyze remech patterns and pilot survival.

3. **Echo Duplicate** (`EchoDuplicateStart` + `EchoDuplicateEnd` tables): Echo's ultimate lets her copy any enemy hero temporarily. We analyze who Echo players target and how long duplicates last.

### Why This Matters
These hero-specific events are often the difference between winning and losing a fight. A well-timed Mercy rez can flip a 4v5 back to even. A D.Va who survives in pilot form and gets remech is a second life for the team's tank. Echo duplicating the right target can swing a fight entirely.

> **For coaches**: Understanding these micro-events helps you evaluate player decision-making beyond raw stats.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_csv, load_kills, load_matches
from src.preprocessing import (
    determine_match_winner, HERO_ROLES, enrich_kills_with_match_info
)
from src.fight_detection import detect_fights, get_fight_kills
from src.visualization import setup_style, OW_COLORS, OW_PALETTE, ROLE_COLORS, save_fig

setup_style()
pd.set_option('display.max_columns', 30)

## 1. Load All Required Data

In [ ]:
# Core tables
kills = load_kills()
match_start, match_end = load_matches()
matches = determine_match_winner(match_end, match_start)

# Hero-specific event tables
mercy_rez = load_csv('MercyRez')
dva_remech = load_csv('DvaRemech')
remech_charged = load_csv('RemechCharged')
echo_dup_start = load_csv('EchoDuplicateStart')
echo_dup_end = load_csv('EchoDuplicateEnd')

# Detect fights for rez impact analysis
valid_kills = kills[
    (kills['attacker_team'] != kills['victim_team']) &
    (kills['attacker_name'] != kills['victim_name'])
].copy()
fights = detect_fights(valid_kills)

print("Hero-Specific Event Counts:")
print(f"  Mercy Resurrects: {len(mercy_rez):,}")
print(f"  D.Va Remechs:     {len(dva_remech):,}")
print(f"  Remech Charged:   {len(remech_charged):,}")
print(f"  Echo Dup Start:   {len(echo_dup_start):,}")
print(f"  Echo Dup End:     {len(echo_dup_end):,}")
print(f"\nFights detected: {len(fights):,}")

---
## 2. Mercy Resurrect Analysis

Mercy's Resurrect ("rez") is a 30-second cooldown ability that brings a dead teammate back to life. It's one of the highest-impact single abilities in the game, but it also puts Mercy at extreme risk (1.75s cast time, vulnerable and slow).

### Key Questions
- How many rezzes happen per match?
- Which heroes get rezzed the most?
- When during fights do rezzes happen?
- Do rezzes change fight outcomes?

In [ ]:
# Basic rez statistics
print(f"Total resurrects: {len(mercy_rez):,}")
print(f"Unique Mercy players performing rezzes: {mercy_rez['resurrecter_player'].nunique()}")
print(f"Matches with at least one rez: {mercy_rez['MapDataId'].nunique()}")
print(f"Rezzes per match (among matches with rez): {mercy_rez.groupby('MapDataId').size().mean():.1f}")
print()

# Rezzes per match distribution
rez_per_match = mercy_rez.groupby('MapDataId').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rez_per_match, bins=range(1, rez_per_match.max() + 2),
             color=OW_COLORS['gold'], edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
axes[0].set_xlabel('Resurrects per Match')
axes[0].set_ylabel('Number of Matches')
axes[0].set_title('Distribution of Resurrects per Match')
axes[0].axvline(rez_per_match.mean(), color=OW_COLORS['red'], linestyle='--',
                label=f'Mean: {rez_per_match.mean():.1f}')
axes[0].legend()

# Rez timing within matches
rez_times = mercy_rez['match_time'].clip(upper=900)
axes[1].hist(rez_times, bins=45, color=OW_COLORS['teal'],
             edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
axes[1].set_xlabel('Match Time (seconds)')
axes[1].set_ylabel('Number of Rezzes')
axes[1].set_title('When Do Rezzes Happen?')

plt.tight_layout()
save_fig(fig, '08_rez_overview')
plt.show()

In [ ]:
# Most-rezzed heroes
rez_heroes = mercy_rez['resurrectee_hero'].value_counts()
rez_heroes_top = rez_heroes.head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) 
          for h in rez_heroes_top.index]
bars = ax.barh(rez_heroes_top.index[::-1], rez_heroes_top.values[::-1], color=colors[::-1])

for bar, val in zip(bars, rez_heroes_top.values[::-1]):
    ax.text(val + rez_heroes_top.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/len(mercy_rez)*100:.1f}%)', va='center', fontsize=9,
            color=OW_COLORS['white'])

ax.set_xlabel('Times Resurrected')
ax.set_title('Most Resurrected Heroes', fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in ROLE_COLORS.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '08_most_rezzed_heroes')
plt.show()

# Rez by role
rez_roles = mercy_rez['resurrectee_hero'].map(HERO_ROLES).value_counts()
print("\nRezzes by role:")
for role, count in rez_roles.items():
    print(f"  {role}: {count:,} ({count/len(mercy_rez)*100:.1f}%)")

In [ ]:
# Rez timing within fights: do rezzes happen early, mid, or late in fights?
# Join rezzes with fights by MapDataId and match_time overlap
rez_in_fights = []

for _, fight in fights.iterrows():
    fight_rezzes = mercy_rez[
        (mercy_rez['MapDataId'] == fight['MapDataId']) &
        (mercy_rez['match_time'] >= fight['fight_start'] - 5) &  # 5s grace before fight
        (mercy_rez['match_time'] <= fight['fight_end'] + 10)    # 10s grace after fight
    ].copy()
    
    if len(fight_rezzes) > 0:
        fight_rezzes = fight_rezzes.copy()
        fight_rezzes['fight_id'] = fight['fight_id']
        fight_rezzes['fight_winner'] = fight['winner']
        fight_rezzes['rez_timing'] = fight_rezzes['match_time'] - fight['fight_start']
        fight_rezzes['fight_duration'] = fight['fight_duration']
        # Relative timing: 0 = fight start, 1 = fight end
        if fight['fight_duration'] > 0:
            fight_rezzes['relative_timing'] = fight_rezzes['rez_timing'] / fight['fight_duration']
        else:
            fight_rezzes['relative_timing'] = 0.5
        rez_in_fights.append(fight_rezzes)

if len(rez_in_fights) > 0:
    rez_fight_df = pd.concat(rez_in_fights, ignore_index=True)
    
    print(f"Rezzes occurring during/near fights: {len(rez_fight_df):,} ({len(rez_fight_df)/len(mercy_rez)*100:.1f}% of all rezzes)")
    print(f"Fights with at least one rez: {rez_fight_df['fight_id'].nunique():,} ({rez_fight_df['fight_id'].nunique()/len(fights)*100:.1f}% of fights)")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Absolute timing from fight start
    axes[0].hist(rez_fight_df['rez_timing'].clip(-5, 30), bins=35,
                 color=OW_COLORS['gold'], edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
    axes[0].axvline(0, color=OW_COLORS['red'], linestyle='--', label='Fight start')
    axes[0].set_xlabel('Seconds from Fight Start')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Rez Timing Relative to Fight Start')
    axes[0].legend()
    
    # Rez team win rate: does the rezzed team win the fight?
    rez_fight_df['rez_team_won'] = rez_fight_df['resurrecter_team'] == rez_fight_df['fight_winner']
    valid_rez = rez_fight_df[rez_fight_df['fight_winner'] != 'Draw']
    rez_win_rate = valid_rez['rez_team_won'].mean()
    
    categories = ['Rez Team\nWins Fight', 'Rez Team\nLoses Fight']
    values = [rez_win_rate * 100, (1 - rez_win_rate) * 100]
    bar_colors = [OW_COLORS['green'], OW_COLORS['red']]
    bars = axes[1].bar(categories, values, color=bar_colors, width=0.5)
    for bar, val in zip(bars, values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.1f}%', ha='center', fontsize=14, fontweight='bold',
                     color=OW_COLORS['white'])
    axes[1].set_ylabel('Percentage')
    axes[1].set_title(f'Fight Win Rate for Rez Team (n={len(valid_rez):,})')
    axes[1].set_ylim(0, 100)
    
    plt.tight_layout()
    save_fig(fig, '08_rez_fight_impact')
    plt.show()
else:
    print("No rezzes detected during fights (data may be limited).")

### Rez Impact Summary

The rez fight win rate tells us how often the team that uses rez ends up winning the fight. Compare this to the ~50% baseline (since the rez team is usually behind by one kill when they rez). If the rez team wins significantly more than 50%, it confirms that rez is a high-value ability that can swing fights.

---
## 3. D.Va Remech Analysis

D.Va is unique among tanks: when her mech is destroyed, she enters "pilot form" (Baby D.Va) — a squishy 150 HP hero with a pistol. She can earn her mech back by:
1. **Charging remech** via dealing damage in pilot form (`RemechCharged`)
2. **Calling mech** once charged (`DvaRemech`)

Surviving in pilot form and getting back into mech is a critical skill for D.Va players. Teams that lose their tank to pilot death effectively lose the fight.

### Key Questions
- How often do D.Va players successfully remech?
- How long does it take to charge remech?
- What's the survival rate in pilot form (remechs vs pilot deaths)?

In [ ]:
# Basic remech statistics
print(f"Total D.Va remechs: {len(dva_remech):,}")
print(f"Total remech charges: {len(remech_charged):,}")
print(f"Unique D.Va players (remech): {dva_remech['player_name'].nunique()}")
print(f"Matches with D.Va remech: {dva_remech['MapDataId'].nunique()}")
print()

# Remechs per match
remech_per_match = dva_remech.groupby('MapDataId').size()
print(f"Remechs per match (avg): {remech_per_match.mean():.1f}")
print(f"Remechs per match (median): {remech_per_match.median():.0f}")

In [ ]:
# D.Va pilot survival analysis
# A D.Va player who remechs has "survived" pilot form
# A D.Va player who dies in pilot form (victim_hero = D.Va in Kill table with pilot-form indicators)
# We approximate: count remechs vs D.Va deaths that occur between demeching and remeching

# D.Va deaths (as victim) - these include both mech and pilot deaths
dva_deaths = valid_kills[valid_kills['victim_hero'] == 'D.Va'].copy()
dva_remech_events = dva_remech.copy()

# Per-player per-match remech counts
remechs_per_player_match = dva_remech.groupby(['MapDataId', 'player_name']).size().reset_index(name='remechs')
dva_deaths_per_player_match = dva_deaths.groupby(['MapDataId', 'victim_name']).size().reset_index(name='deaths')
dva_deaths_per_player_match = dva_deaths_per_player_match.rename(columns={'victim_name': 'player_name'})

# Merge
dva_survival = remechs_per_player_match.merge(
    dva_deaths_per_player_match, on=['MapDataId', 'player_name'], how='outer'
).fillna(0)

total_remechs = dva_survival['remechs'].sum()
total_dva_deaths = dva_survival['deaths'].sum()

print(f"\nD.Va Pilot Form Survival Indicators:")
print(f"  Total remechs (survived pilot): {total_remechs:.0f}")
print(f"  Total D.Va deaths (all forms):  {total_dva_deaths:.0f}")
print(f"  Remech-to-death ratio: {total_remechs / max(total_dva_deaths, 1):.2f}")
print()
print("Note: D.Va deaths include mech deaths (demech events), so the ratio")
print("represents how many times D.Va gets back into mech per time she dies overall.")

In [ ]:
# Remech timing analysis: when during matches do remechs happen?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Remech timing in match
axes[0].hist(dva_remech['match_time'].clip(upper=900), bins=45,
             color=OW_COLORS['blue'], edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
axes[0].set_xlabel('Match Time (seconds)')
axes[0].set_ylabel('Number of Remechs')
axes[0].set_title('When Do D.Va Remechs Happen?')

# Remechs per match distribution
axes[1].hist(remech_per_match, bins=range(1, remech_per_match.max() + 2),
             color=OW_COLORS['teal'], edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
axes[1].set_xlabel('Remechs per Match')
axes[1].set_ylabel('Number of Matches')
axes[1].set_title('Distribution of Remechs per Match')
axes[1].axvline(remech_per_match.mean(), color=OW_COLORS['red'], linestyle='--',
                label=f'Mean: {remech_per_match.mean():.1f}')
axes[1].legend()

plt.tight_layout()
save_fig(fig, '08_dva_remech_patterns')
plt.show()

In [ ]:
# Charge time analysis: how long does it take to charge remech?
# Match remech_charged events to the subsequent dva_remech event for same player/match
charge_to_remech = []

for (map_id, player), group in dva_remech.groupby(['MapDataId', 'player_name']):
    charges = remech_charged[
        (remech_charged['MapDataId'] == map_id) & 
        (remech_charged['player_name'] == player)
    ].sort_values('match_time')
    remechs = group.sort_values('match_time')
    
    # For each remech, find the most recent charge event before it
    for _, rm in remechs.iterrows():
        prior_charges = charges[charges['match_time'] < rm['match_time']]
        if len(prior_charges) > 0:
            last_charge = prior_charges.iloc[-1]
            charge_to_remech.append({
                'MapDataId': map_id,
                'player_name': player,
                'charge_time': last_charge['match_time'],
                'remech_time': rm['match_time'],
                'charge_duration': rm['match_time'] - last_charge['match_time'],
            })

if len(charge_to_remech) > 0:
    charge_df = pd.DataFrame(charge_to_remech)
    # Filter reasonable durations (0-60s; longer likely indicates missed events)
    charge_df = charge_df[(charge_df['charge_duration'] > 0) & (charge_df['charge_duration'] <= 60)]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(charge_df['charge_duration'], bins=30, color=OW_COLORS['purple'],
            edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
    ax.axvline(charge_df['charge_duration'].median(), color=OW_COLORS['gold'], linestyle='--',
               label=f'Median: {charge_df["charge_duration"].median():.1f}s')
    ax.set_xlabel('Time from Charge to Remech (seconds)')
    ax.set_ylabel('Count')
    ax.set_title('D.Va: Time from Remech Charged to Call Mech', fontsize=14, fontweight='bold')
    ax.legend()
    
    plt.tight_layout()
    save_fig(fig, '08_dva_charge_to_remech')
    plt.show()
    
    print(f"Charge-to-remech pairs found: {len(charge_df):,}")
    print(f"Mean time: {charge_df['charge_duration'].mean():.1f}s")
    print(f"Median time: {charge_df['charge_duration'].median():.1f}s")
else:
    print("Not enough charge-remech pairs found for timing analysis.")

---
## 4. Echo Duplicate Analysis

Echo's ultimate ability, **Duplicate**, lets her copy any enemy hero for a short duration (15 seconds). During this time she becomes that hero with all their abilities and can charge *their* ultimate extremely quickly (6.5x rate).

### Key Questions
- Which heroes do Echo players choose to duplicate?
- How long do duplicates last? (Ending early might mean Echo was killed)
- Are there preferred duplicate targets by map type or team?

In [ ]:
# Basic Echo duplicate stats
print(f"Total Echo Duplicate starts: {len(echo_dup_start):,}")
print(f"Total Echo Duplicate ends: {len(echo_dup_end):,}")
print(f"Unique Echo players: {echo_dup_start['player_name'].nunique()}")
print(f"Matches with Echo Duplicate: {echo_dup_start['MapDataId'].nunique()}")
print()

# Most duplicated heroes
dup_targets = echo_dup_start['hero_duplicated'].value_counts()
print("Top 15 most duplicated heroes:")
for hero, count in dup_targets.head(15).items():
    role = HERO_ROLES.get(hero, 'Unknown')
    pct = count / len(echo_dup_start) * 100
    print(f"  {hero} ({role}): {count} ({pct:.1f}%)")

In [ ]:
# Visualize duplicate targets
dup_targets_top = dup_targets.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart of duplicate targets
colors = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) 
          for h in dup_targets_top.index]
bars = axes[0].barh(dup_targets_top.index[::-1], dup_targets_top.values[::-1], color=colors[::-1])

for bar, val in zip(bars, dup_targets_top.values[::-1]):
    axes[0].text(val + dup_targets_top.max() * 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val}', va='center', fontsize=9, color=OW_COLORS['white'])

axes[0].set_xlabel('Times Duplicated')
axes[0].set_title('Most Duplicated Heroes by Echo', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in ROLE_COLORS.items()]
axes[0].legend(handles=legend_elements, loc='lower right')

# Duplicate targets by role (pie chart)
dup_roles = echo_dup_start['hero_duplicated'].map(HERO_ROLES).value_counts()
role_colors_pie = [ROLE_COLORS.get(r, OW_COLORS['light_gray']) for r in dup_roles.index]
axes[1].pie(dup_roles.values, labels=dup_roles.index, colors=role_colors_pie,
            autopct='%1.1f%%', textprops={'color': OW_COLORS['white']}, startangle=90,
            pctdistance=0.8)
axes[1].set_title('Duplicate Targets by Role', fontsize=13, fontweight='bold')

plt.tight_layout()
save_fig(fig, '08_echo_duplicate_targets')
plt.show()

In [ ]:
# Duplicate duration analysis
# Match start and end events by ultimate_id and player
dup_merged = echo_dup_start.merge(
    echo_dup_end[['MapDataId', 'player_name', 'ultimate_id', 'match_time']],
    on=['MapDataId', 'player_name', 'ultimate_id'],
    how='inner',
    suffixes=('_start', '_end')
)

dup_merged['duration'] = dup_merged['match_time_end'] - dup_merged['match_time_start']

# Filter to reasonable durations (0-20s; Echo dup lasts max 15s)
dup_merged_valid = dup_merged[(dup_merged['duration'] > 0) & (dup_merged['duration'] <= 20)]

print(f"Matched duplicate start/end pairs: {len(dup_merged_valid):,}")
print(f"Mean duration: {dup_merged_valid['duration'].mean():.1f}s")
print(f"Median duration: {dup_merged_valid['duration'].median():.1f}s")
print(f"Full duration (>=14s): {(dup_merged_valid['duration'] >= 14).sum()} ({(dup_merged_valid['duration'] >= 14).mean()*100:.1f}%)")
print(f"Killed early (<10s): {(dup_merged_valid['duration'] < 10).sum()} ({(dup_merged_valid['duration'] < 10).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration histogram
axes[0].hist(dup_merged_valid['duration'], bins=20, color=OW_COLORS['purple'],
             edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
axes[0].axvline(15, color=OW_COLORS['gold'], linestyle='--', label='Max duration (15s)')
axes[0].axvline(dup_merged_valid['duration'].median(), color=OW_COLORS['red'], linestyle='--',
                label=f'Median: {dup_merged_valid["duration"].median():.1f}s')
axes[0].set_xlabel('Duplicate Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Echo Duplicate Duration')
axes[0].legend()

# Duration by target hero (top 8)
top_targets = dup_targets.head(8).index.tolist()
dur_by_target = dup_merged_valid[dup_merged_valid['hero_duplicated'].isin(top_targets)]
target_durations = dur_by_target.groupby('hero_duplicated')['duration'].mean().sort_values(ascending=True)

colors = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) 
          for h in target_durations.index]
axes[1].barh(target_durations.index, target_durations.values, color=colors)
axes[1].set_xlabel('Average Duration (seconds)')
axes[1].set_title('Avg Duplicate Duration by Target Hero')
axes[1].axvline(15, color=OW_COLORS['gold'], linestyle=':', alpha=0.5)

plt.tight_layout()
save_fig(fig, '08_echo_duplicate_duration')
plt.show()

---
## 5. Summary & Coaching Implications

### Mercy Resurrect

| Finding | Implication |
|---------|------------|
| Rez targets are role-dependent | Tanks and key DPS get priority — coaches should set rez priority lists |
| Rez timing within fights matters | Early rezzes (before fight is decided) are more impactful |
| Rez team win rate | Confirms rez is a fight-swinging ability worth tracking |

### D.Va Remech

| Finding | Implication |
|---------|------------|
| Remech frequency varies by match | D.Va's survivability directly impacts team sustain |
| Charge-to-remech time | Faster remechs mean less time without a tank — a key D.Va skill |
| Pilot survival is critical | Losing D.Va in pilot form is effectively losing a teamfight |

### Echo Duplicate

| Finding | Implication |
|---------|------------|
| Target preferences are clear | Echo players strongly prefer certain heroes — meta-dependent |
| Duration varies by target | Some duplicates get shut down fast — target choice affects survival |
| Tank duplication is popular | Duplicating tanks gives maximum value via health pool and disruption |

### What ScrimSight Should Build

1. **Rez tracker**: Show Mercy players their rez timing, targets, and fight impact
2. **D.Va survival dashboard**: Track pilot deaths vs remechs as a D.Va-specific KPI
3. **Echo duplicate log**: Show duplicate target choices and their outcomes
4. **Hero-specific event alerts**: Flag unusually good/bad hero-specific performance per match